# 02_mlflow_model_registry

In [27]:
import sys
from pathlib import Path
import mlflow
import numpy as np
import pandas as pd

sys.executable

# Repo root (notebook is in /notebooks)
PROJECT_ROOT = Path.cwd().parent

# Dataset path
csv_path = PROJECT_ROOT / "data" / "raw" / "adult-census.csv"

# Everything under repo/mlflow/
MLFLOW_DIR = PROJECT_ROOT / "mlflow"
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)

MLFLOW_DB = (MLFLOW_DIR / "mlflow.db").resolve()
ARTIFACT_ROOT = (MLFLOW_DIR / "artifacts").resolve()
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB.as_posix()}")

# ✅ Keep the SAME artifact root for all experiments
# so everything lands under repo/mlflow/artifacts/
TRACKING_EXPERIMENT = "adult_census_tracking"
REGISTRY_EXPERIMENT  = "adult_census_registry"

# Create experiments if missing
for exp_name in [TRACKING_EXPERIMENT, REGISTRY_EXPERIMENT]:
    exp = mlflow.get_experiment_by_name(exp_name)
    if exp is None:
        mlflow.create_experiment(
            name=exp_name,
            artifact_location=ARTIFACT_ROOT.as_uri(),
        )

# Default experiment for this notebook (registry ops)
mlflow.set_experiment(REGISTRY_EXPERIMENT)

# Reports outputs (local, then logged to MLflow as artifacts)
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# (Optional) Dedicated folder for registry-related reports
REGISTRY_REPORTS_DIR = REPORTS_DIR / "registry"
REGISTRY_REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT       =", PROJECT_ROOT)
print("csv_path           =", csv_path)
print("exists?            =", csv_path.exists())
print("MLFLOW_DIR         =", MLFLOW_DIR)
print("TRACKING_URI       =", mlflow.get_tracking_uri())
print("MLFLOW_DB          =", MLFLOW_DB)
print("ARTIFACT_ROOT      =", ARTIFACT_ROOT)
print("REPORTS_DIR        =", REPORTS_DIR)
print("REGISTRY_REPORTS   =", REGISTRY_REPORTS_DIR)
print("EXPERIMENT (active)=", mlflow.get_experiment_by_name(REGISTRY_EXPERIMENT).name)

PROJECT_ROOT       = h:\Documents\2. Perso\github\mlflow
csv_path           = h:\Documents\2. Perso\github\mlflow\data\raw\adult-census.csv
exists?            = True
MLFLOW_DIR         = h:\Documents\2. Perso\github\mlflow\mlflow
TRACKING_URI       = sqlite:///H:/Documents/2. Perso/github/mlflow/mlflow/mlflow.db
MLFLOW_DB          = H:\Documents\2. Perso\github\mlflow\mlflow\mlflow.db
ARTIFACT_ROOT      = H:\Documents\2. Perso\github\mlflow\mlflow\artifacts
REPORTS_DIR        = h:\Documents\2. Perso\github\mlflow\reports
REGISTRY_REPORTS   = h:\Documents\2. Perso\github\mlflow\reports\registry
EXPERIMENT (active)= adult_census_registry


## Section 1 — Setup MLflow Registry

### Tracking ≠ Registry
- **Tracking** logs *runs* (params, metrics, artifacts): “what did I try and what happened?”
- **Registry** manages the *model lifecycle* (model name, versions, stages): “which model should be used in Staging/Production?”

### Why use a Registry even locally?
Because it gives you a **stable reference** (`models:/.../Production`) instead of hardcoding a `run_id`, and lets you practice **versioning, promotions, and rollbacks** in a production-like workflow, even with a local SQLite setup.

In [2]:
# Section 1 — Setup MLflow Registry (on top of your existing Tracking setup)

from mlflow.tracking import MlflowClient

REGISTRY_EXPERIMENT = "adult_census_registry"
mlflow.set_experiment(REGISTRY_EXPERIMENT)

client = MlflowClient()
MODEL_NAME = "adult_census_classifier"

print("REGISTRY_EXPERIMENT =", REGISTRY_EXPERIMENT)
print("TRACKING_URI        =", mlflow.get_tracking_uri())
print("MODEL_NAME          =", MODEL_NAME)

REGISTRY_EXPERIMENT = adult_census_registry
TRACKING_URI        = sqlite:///H:/Documents/2. Perso/github/mlflow/mlflow/mlflow.db
MODEL_NAME          = adult_census_classifier


## Section 2 — List versions (state of the world)

In [3]:
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
print("Found versions:", len(versions))

if not versions:
    raise RuntimeError(f"No versions found for model '{MODEL_NAME}'")

for v in versions:
    print(f"v{v.version} | stage={v.current_stage} | run_id={v.run_id}")

latest_version = str(max(int(v.version) for v in versions))
print("Latest version:", latest_version)

2026/01/21 10:46:00 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/01/21 10:46:00 INFO mlflow.store.db.utils: Updating database tables
2026/01/21 10:46:00 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/01/21 10:46:00 INFO alembic.runtime.migration: Will assume non-transactional DDL.


Found versions: 6
v6 | stage=None | run_id=13ad09f77b8d411ead1312e2eb231e2e
v5 | stage=None | run_id=74c69fa04600460daa23bfb4f5298a90
v4 | stage=None | run_id=a5f9cfe2f3aa4534b8d12fc1112fba01
v3 | stage=None | run_id=2185a9e0a2a545fa8902c04d0c05b043
v2 | stage=Archived | run_id=5da6635ebf7a44d68c54ad54ca9d0bb4
v1 | stage=Production | run_id=6adf9bfc23544540b515f095090d1f32
Latest version: 6


## Section 3 — Promote latest → Staging (logged)

In [4]:
with mlflow.start_run(run_name=f"promote_{MODEL_NAME}_v{latest_version}_to_staging"):
    client.transition_model_version_stage(
        name=MODEL_NAME,
        version=latest_version,
        stage="Staging",
    )
    mlflow.set_tag("model_name", MODEL_NAME)
    mlflow.set_tag("model_version", latest_version)
    mlflow.set_tag("action", "promote_to_staging")

print("Promoted to Staging:", latest_version)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_7316\3941252493.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


Promoted to Staging: 6


## Section 4 — Promote latest → Production + load from Production (logged)

In [5]:
import mlflow.pyfunc

with mlflow.start_run(run_name=f"promote_{MODEL_NAME}_v{latest_version}_to_production_and_load"):
    client.transition_model_version_stage(
        name=MODEL_NAME,
        version=latest_version,
        stage="Production",
        archive_existing_versions=True,
    )
    mlflow.set_tag("model_name", MODEL_NAME)
    mlflow.set_tag("model_version", latest_version)
    mlflow.set_tag("action", "promote_to_production")

    model_prod = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}/Production")
    mlflow.set_tag("loaded_uri", f"models:/{MODEL_NAME}/Production")

print(f"✓ {MODEL_NAME} v{latest_version} promoted to Production")
print("✓ Loaded model from Production")

C:\Users\Administrator\AppData\Local\Temp\ipykernel_7316\1752371037.py:4: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


✓ adult_census_classifier v6 promoted to Production
✓ Loaded model from Production


## Section 5 — Sanity check inference (self-contained)

In [6]:
# Section 5 — Sanity check inference
# ------------------------------------------------------------------------------
# Goal:
# - Reproduce the exact same dataset preparation as in notebook 01_baseline_sklearn_pipeline.ipynb
# - Load the model from the MLflow Registry "Production" stage
# - Run a quick inference sanity check on a few rows (and optionally on the full test set)

import pandas as pd
import mlflow.pyfunc
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

MODEL_NAME = "adult_census_classifier"

# 1) Load the model from the Registry (Production stage)
model_prod = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}/Production")
print(f"Loaded model from: models:/{MODEL_NAME}/Production")

# 2) Load dataset and apply the exact same preprocessing as Notebook #1
adult_census = pd.read_csv(csv_path)

# Notebook #1 removes this column (keep it identical)
adult_census = adult_census.drop(columns="education.num")

# Define target + features exactly like Notebook #1
target_name = "income"
target = adult_census[target_name]
data = adult_census.drop(columns=[target_name])

# Encode target with the exact mapping used in Notebook #1
target_map = {"<=50K": 0, ">50K": 1}
target_enc = target.map(target_map).astype("int64")

# 3) Use the same train/test split configuration (random_state + stratify)
data_train, data_test, target_train, target_test = train_test_split(
    data,
    target_enc,
    test_size=0.2,
    random_state=42,
    stratify=target_enc,
)

# 4) Sanity check: predict on a small sample
preds_sample = model_prod.predict(data_test.head(10))
print("Sample preds :", np.asarray(preds_sample))
print("Sample y_true:", target_test.head(10).to_numpy())

# 5) Quick overall accuracy on the test split
preds_all = model_prod.predict(data_test)
acc = accuracy_score(target_test, preds_all)
print(f"Sanity test accuracy: {acc:.4f}")

Loaded model from: models:/adult_census_classifier/Production
Sample preds : [0 0 1 1 0 1 0 0 0 0]
Sample y_true: [0 0 1 1 0 0 0 0 0 0]
Sanity test accuracy: 0.8675


## Section 6 — Create a v2 (new model version) and log/register it

In [7]:
# Section 6 — Train + Register a v2 (LogReg) as a new Model Version in the Registry
# ------------------------------------------------------------------------------
# Goal:
# - Train a new sklearn pipeline (v2)
# - Log metrics + params
# - Register it under the SAME Registered Model name (MODEL_NAME)
#   => this creates a NEW model version (v2) in the MLflow Registry

import numpy as np
import mlflow
import mlflow.sklearn

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

# We reuse the exact same split variables from Notebook #1 / Section 5:
# data_train, data_test, target_train, target_test

# 1) Detect numerical vs categorical columns on the training set
num_cols = data_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in data_train.columns if c not in num_cols]

# 2) Build preprocessing pipelines
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False)),
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop",
)

# 3) Define v2 model (Logistic Regression)
v2_pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", LogisticRegression(max_iter=500)),
])

# 4) Train + log + register
with mlflow.start_run(run_name="train_register_v2_logreg") as run:
    # Fit
    v2_pipeline.fit(data_train, target_train)

    # Predict
    y_pred = v2_pipeline.predict(data_test)

    # Metrics
    acc = accuracy_score(target_test, y_pred)
    mlflow.log_metric("test_accuracy", acc)

    # AUC (only for binary classification + proba available)
    if hasattr(v2_pipeline, "predict_proba") and len(np.unique(target_test)) == 2:
        y_proba = v2_pipeline.predict_proba(data_test)[:, 1]
        auc = roc_auc_score(target_test, y_proba)
        mlflow.log_metric("test_roc_auc", auc)

    # Params (useful for UI comparison)
    mlflow.log_param("model_family", "logreg")
    mlflow.log_param("max_iter", 500)
    mlflow.log_param("n_num_cols", len(num_cols))
    mlflow.log_param("n_cat_cols", len(cat_cols))

    # Register as a NEW version under the same Registered Model
    mlflow.sklearn.log_model(
        sk_model=v2_pipeline,
        artifact_path="model",
        registered_model_name=MODEL_NAME,
    )

print("✓ v2 logged + registered")
print("Run ID:", run.info.run_id)
print("Model:", MODEL_NAME)
print("Check MLflow UI > Models to see the new version")

2026/01/21 10:49:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'adult_census_classifier' already exists. Creating a new version of this model...
Created version '7' of model 'adult_census_classifier'.


✓ v2 logged + registered
Run ID: 82ae9645b834457ea7334776bc6ac38e
Model: adult_census_classifier
Check MLflow UI > Models to see the new version


## Section 7 — Find v2 version number, promote it to Production, then rollback to v1

In [8]:
# Section 7 — Promote v2 to Production, then rollback to v1 (Registry lifecycle)
# ------------------------------------------------------------------------------
# Goal:
# - Fetch all model versions from the Registry
# - Identify v1 (oldest) and v2 (latest)
# - Promote v2 to Production (and archive existing Production)
# - Rollback to v1 (promote v1 back to Production)
#
# This section does NOT train anything.
# It only manipulates lifecycle stages in the MLflow Model Registry.

from mlflow.tracking import MlflowClient

client = MlflowClient()

# Fetch all model versions for the registered model
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
all_versions = sorted({int(v.version) for v in versions})

print("All versions found:", all_versions)

# Safety check: we need at least 2 versions to demonstrate promotion + rollback
if len(all_versions) < 2:
    raise RuntimeError(
        f"Need at least 2 model versions for '{MODEL_NAME}' to run promotion + rollback."
    )

# v1 = oldest, v2 = latest
v1 = str(min(all_versions))
v2 = str(max(all_versions))

print("v1 (oldest) =", v1)
print("v2 (latest) =", v2)

All versions found: [1, 2, 3, 4, 5, 6, 7]
v1 (oldest) = 1
v2 (latest) = 7


In [9]:
# Promote v2 to Production (archive any currently-Production versions)
# ------------------------------------------------------------------------------
# We log this operation as a dedicated run in the "adult_census_registry" experiment,
# to keep lifecycle actions separated from training runs.

import mlflow

mlflow.set_experiment("adult_census_registry")

with mlflow.start_run(run_name=f"promote_{MODEL_NAME}_v{v2}_to_production"):
    client.transition_model_version_stage(
        name=MODEL_NAME,
        version=v2,
        stage="Production",
        archive_existing_versions=True,
    )

    # Minimal metadata for traceability
    mlflow.set_tag("model_name", MODEL_NAME)
    mlflow.set_tag("model_version", v2)
    mlflow.set_tag("action", "promote_to_production")

print(f"✓ {MODEL_NAME} v{v2} is now Production (previous Production version archived)")

C:\Users\Administrator\AppData\Local\Temp\ipykernel_7316\860144185.py:11: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


✓ adult_census_classifier v7 is now Production (previous Production version archived)


In [10]:
# Rollback to v1 (promote v1 back to Production)
# ------------------------------------------------------------------------------
# This simulates a real incident response: v2 had issues in Production,
# so we revert to the last known good version (v1).

with mlflow.start_run(run_name=f"rollback_{MODEL_NAME}_to_v{v1}"):
    client.transition_model_version_stage(
        name=MODEL_NAME,
        version=v1,
        stage="Production",
        archive_existing_versions=True,
    )

    # Minimal metadata for traceability
    mlflow.set_tag("model_name", MODEL_NAME)
    mlflow.set_tag("model_version", v1)
    mlflow.set_tag("action", "rollback_to_v1")

print(f"✓ Rollback complete: {MODEL_NAME} v{v1} is back to Production")

C:\Users\Administrator\AppData\Local\Temp\ipykernel_7316\2389825216.py:7: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


✓ Rollback complete: adult_census_classifier v1 is back to Production


In [11]:
# Smoke test after rollback: load the Production model from the Registry
# ------------------------------------------------------------------------------
# In a real service, inference code should only ever depend on:
#   models:/<model_name>/Production
# not on a run_id.

import mlflow.pyfunc

model_prod = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}/Production")
print(f"✓ Loaded model from: models:/{MODEL_NAME}/Production")

✓ Loaded model from: models:/adult_census_classifier/Production


## Section 8 — Champion / Challenger (aliases, production-friendly)

Stages (Staging/Production) are useful, but **aliases** are often cleaner in production:

- `@champion` = the model currently serving in production
- `@challenger` = the best candidate next model

This enables:
- stable URIs (`models:/name@champion`)
- easy swaps (`champion ← challenger`)
- A/B testing patterns (future)

In [12]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
MODEL_NAME = "adult_census_classifier"

def safe_set_alias(model_name: str, alias: str, version: str):
    """Set a registered model alias if supported by the installed MLflow version."""
    if hasattr(client, "set_registered_model_alias"):
        client.set_registered_model_alias(model_name, alias, version)
        return True
    return False

def safe_get_by_alias(model_name: str, alias: str):
    """Get model version by alias if supported."""
    if hasattr(client, "get_model_version_by_alias"):
        return client.get_model_version_by_alias(model_name, alias)
    return None

# We already computed v1/v2 in your Section 7.
# If not present in memory (fresh kernel), recompute quickly:
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
all_versions = sorted({int(v.version) for v in versions})
if len(all_versions) < 2:
    raise RuntimeError("Need at least 2 versions to set champion/challenger.")

v1 = str(min(all_versions))
v2 = str(max(all_versions))

print("Available versions:", all_versions)
print("v1 =", v1, "(oldest)")
print("v2 =", v2, "(latest)")

# Convention:
# - champion starts as the current Production version
# - challenger starts as the latest version
# If aliases exist, we set them. Otherwise, we rely on stages.
set_ok = safe_set_alias(MODEL_NAME, "champion", v1)
set_ok &= safe_set_alias(MODEL_NAME, "challenger", v2)

if set_ok:
    print("✓ Aliases set:")
    print("  champion   -> v", v1)
    print("  challenger -> v", v2)
else:
    print("⚠️ Aliases not supported by your MLflow version. We'll rely on stages (Production/Staging).")

Available versions: [1, 2, 3, 4, 5, 6, 7]
v1 = 1 (oldest)
v2 = 7 (latest)
✓ Aliases set:
  champion   -> v 1
  challenger -> v 7


## Section 9 — How to choose a Challenger (the real rule)

A challenger is **not** "the latest model".
A challenger is **the best candidate that beats the champion** under a clear policy.

### Recommended policy (simple but real)
Primary metric (imbalanced dataset):
- **PR-AUC** (Average Precision)

Promotion gate:
- Challenger PR-AUC must be **>= Champion PR-AUC + 0.01**
- AND must not degrade Recall by more than **-0.02**
- AND inference latency must not be worse by more than **+25%**

If it passes:
- assign `@challenger`
- optionally swap `@champion` (promotion)

In [14]:
from sklearn.model_selection import train_test_split

# Rebuild a deterministic evaluation split.
# IMPORTANT: the challenger decision must be made on the SAME data split for all models.

TARGET_COL = "income"
RANDOM_STATE = 42
TEST_SIZE = 0.2

data = pd.read_csv(csv_path)

# If you removed a column in notebook 01, do it here too (safe, no crash if absent)
data = data.drop(columns=["education-num"], errors="ignore")

y_raw = data[TARGET_COL].astype(str).str.strip()
X = data.drop(columns=[TARGET_COL])

# Robust encoding (covers '>50K', '>50K.' and potential spacing)
POS_LABELS = {">50K", ">50K.", "1"}
y = y_raw.isin(POS_LABELS).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Eval split ready:")
print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("Positive rate (train):", y_train.mean().round(4))
print("Positive rate (test) :", y_test.mean().round(4))

Eval split ready:
X_train: (26048, 14) X_test: (6513, 14)
Positive rate (train): 0.2408
Positive rate (test) : 0.2407


In [15]:
import time
import json
import os
import mlflow
import mlflow.sklearn

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
)

def load_sklearn_registry_model(model_name: str, version_or_stage: str):
    """
    Load a model from the registry using the sklearn flavor.
    Works with:
      - models:/name/Production
      - models:/name/3
    and (if supported):
      - models:/name@champion
    """
    if version_or_stage.startswith("@"):
        uri = f"models:/{model_name}{version_or_stage}"
    elif version_or_stage in {"Production", "Staging", "Archived", "None"}:
        uri = f"models:/{model_name}/{version_or_stage}"
    else:
        uri = f"models:/{model_name}/{version_or_stage}"

    try:
        model = mlflow.sklearn.load_model(uri)
        return uri, model
    except Exception as e:
        raise RuntimeError(
            f"Could not load sklearn model from registry at '{uri}'. "
            f"Make sure it was logged with mlflow.sklearn.log_model.\nOriginal error: {e}"
        )

def get_predict_proba_or_score(model, X):
    """Return a continuous score for AUC metrics (proba if available, else decision_function)."""
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        return proba[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    raise RuntimeError("Model has neither predict_proba nor decision_function.")

def measure_latency(model, X, n_runs: int = 20, batch_size: int = 512):
    """Approx inference latency per row (ms) using repeated batch prediction."""
    Xb = X.iloc[:batch_size]
    # warmup
    _ = model.predict(Xb)

    t0 = time.perf_counter()
    for _ in range(n_runs):
        _ = model.predict(Xb)
    t1 = time.perf_counter()

    total_preds = n_runs * len(Xb)
    ms_per_row = (t1 - t0) * 1000 / total_preds
    return ms_per_row

def estimate_model_size_bytes(model_name: str, version: str):
    """
    Estimate artifact size by downloading the model artifacts.
    Works locally, and also when artifacts are in remote store.
    """
    # This downloads the whole version folder locally
    local_dir = mlflow.artifacts.download_artifacts(
        artifact_uri=f"models:/{model_name}/{version}"
    )

    total = 0
    for root, _, files in os.walk(local_dir):
        for f in files:
            fp = os.path.join(root, f)
            total += os.path.getsize(fp)
    return total

def evaluate_registry_version(model_name: str, version: str, X_test, y_test):
    uri, model = load_sklearn_registry_model(model_name, version)

    # Continuous scores for AUC metrics
    y_score = get_predict_proba_or_score(model, X_test)
    y_pred = (y_score >= 0.5).astype(int)

    metrics = {
        "roc_auc": float(roc_auc_score(y_test, y_score)),
        "pr_auc": float(average_precision_score(y_test, y_score)),
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "precision": float(precision_score(y_test, y_pred, zero_division=0)),
        "recall": float(recall_score(y_test, y_pred, zero_division=0)),
        "f1": float(f1_score(y_test, y_pred, zero_division=0)),
    }

    latency_ms_per_row = float(measure_latency(model, X_test))
    size_bytes = float(estimate_model_size_bytes(model_name, version))

    info = {
        "model_uri": uri,
        "version": version,
        "metrics": metrics,
        "latency_ms_per_row": latency_ms_per_row,
        "size_bytes": size_bytes,
    }
    return info

## Section 10 — Compare Champion vs Challenger (decision gate)

We evaluate both models on the SAME hold-out test split.

Decision rule:
- Challenger PR-AUC must be >= Champion PR-AUC + 0.01
- Recall drop must be <= 0.02
- Latency increase must be <= +25%

If it passes:
- set alias `@challenger`
- optionally swap `@champion` (promotion)

In [23]:
from mlflow.tracking import MlflowClient
import pandas as pd
import json

client = MlflowClient()

# ---------------------------
# Config: decision constraints
# ---------------------------
MIN_PR_AUC_GAIN = 0.01
MAX_RECALL_DROP = 0.02
MAX_LATENCY_INCREASE = 0.25  # +25%

# ---------------------------
# 1) Evaluate all versions
# ---------------------------
model_versions = client.search_model_versions(f"name='{MODEL_NAME}'")
all_versions = sorted({str(v.version) for v in model_versions}, key=lambda x: int(x))

if len(all_versions) < 2:
    raise RuntimeError("Need at least 2 registered model versions to compare.")

info_by_version = {}
rows = []

for v in all_versions:
    info = evaluate_registry_version(MODEL_NAME, v, X_test, y_test)
    info_by_version[v] = info
    rows.append({
        "version": v,
        "pr_auc": info["metrics"]["pr_auc"],
        "roc_auc": info["metrics"]["roc_auc"],
        "recall": info["metrics"]["recall"],
        "precision": info["metrics"]["precision"],
        "f1": info["metrics"]["f1"],
        "latency_ms_per_row": info["latency_ms_per_row"],
        "size_bytes": info["size_bytes"],
    })

df = pd.DataFrame(rows).sort_values("version", key=lambda s: s.astype(int)).reset_index(drop=True)

# ---------------------------
# 2) Resolve champion (alias > Production > best PR-AUC)
# ---------------------------
alias_champion = safe_get_by_alias(MODEL_NAME, "champion")

if alias_champion:
    champion_version = str(alias_champion.version).strip()
else:
    prod_versions = client.get_latest_versions(MODEL_NAME, stages=["Production"])
    if prod_versions:
        champion_version = str(prod_versions[0].version).strip()
    else:
        champion_version = str(df.sort_values("pr_auc", ascending=False).iloc[0]["version"]).strip()

champion_info = info_by_version[champion_version]
champion = champion_info["metrics"]
champion_latency = champion_info["latency_ms_per_row"]

# ---------------------------
# 3) Choose challenger = BEST MODEL THAT IS DEPLOYABLE
# ---------------------------
# Feasible set = respects your constraints relative to champion
max_latency = champion_latency * (1.0 + MAX_LATENCY_INCREASE)
min_recall = champion["recall"] - MAX_RECALL_DROP

df_candidates = df[df["version"] != champion_version].copy()

df_feasible = df_candidates[
    (df_candidates["latency_ms_per_row"] <= max_latency) &
    (df_candidates["recall"] >= min_recall)
].copy()

if df_feasible.empty:
    # No feasible challenger -> tranche vite
    challenger_version = str(df_candidates.sort_values("pr_auc", ascending=False).iloc[0]["version"]).strip()
    feasible = False
else:
    # Best feasible challenger by PR-AUC (tie-break recall then latency)
    df_feasible = df_feasible.sort_values(
        by=["pr_auc", "recall", "latency_ms_per_row"],
        ascending=[False, False, True],
    )
    challenger_version = str(df_feasible.iloc[0]["version"]).strip()
    feasible = True

# Optional: set alias challenger
if hasattr(client, "set_registered_model_alias"):
    try:
        client.set_registered_model_alias(MODEL_NAME, "challenger", challenger_version)
        print(f"✓ Alias set: challenger -> v{challenger_version}")
    except Exception:
        pass

print("Champion version  :", champion_version)
print("Challenger version:", challenger_version)

# ---------------------------
# 4) Evaluate champion vs challenger
# ---------------------------
challenger_info = info_by_version[challenger_version]
challenger = challenger_info["metrics"]

print("\n=== Champion ===")
print(json.dumps(champion_info, indent=2))

print("\n=== Challenger ===")
print(json.dumps(challenger_info, indent=2))

pr_gain = challenger["pr_auc"] - champion["pr_auc"]
recall_drop = champion["recall"] - challenger["recall"]
lat_increase = (challenger_info["latency_ms_per_row"] / champion_info["latency_ms_per_row"]) - 1.0

# ---------------------------
# 5) Decision (fast + explicit)
# ---------------------------
if not feasible:
    print("\n--- Feasibility check ---")
    print("No deployable challenger found under constraints.")
    print(f"- Max latency allowed : {max_latency:.6f} ms/row")
    print(f"- Min recall allowed  : {min_recall:.6f}")
    print("=> Keeping champion. (We can still keep this as an exploration candidate.)")
    passes = False
else:
    passes = (
        (pr_gain >= MIN_PR_AUC_GAIN)
        and (recall_drop <= MAX_RECALL_DROP)
        and (lat_increase <= MAX_LATENCY_INCREASE)
    )

print("\n--- Decision gate ---")
print(f"PR-AUC champion   : {champion['pr_auc']:.6f}")
print(f"PR-AUC challenger : {challenger['pr_auc']:.6f}")
print(f"PR-AUC gain       : {pr_gain:+.6f} (>= +{MIN_PR_AUC_GAIN})")
print(f"Recall drop       : {recall_drop:+.6f} (<= +{MAX_RECALL_DROP})")
print(f"Latency increase  : {lat_increase:+.2%} (<= +{MAX_LATENCY_INCREASE:.0%})")
print("\nPASS?" , passes)

✓ Alias set: challenger -> v6
Champion version  : 1
Challenger version: 6

=== Champion ===
{
  "model_uri": "models:/adult_census_classifier/1",
  "version": "1",
  "metrics": {
    "roc_auc": 0.9042055725222344,
    "pr_auc": 0.7678645629298936,
    "accuracy": 0.8544449562413634,
    "precision": 0.739938080495356,
    "recall": 0.6096938775510204,
    "f1": 0.6685314685314685
  },
  "latency_ms_per_row": 0.037349882813941804,
  "size_bytes": 6186.0
}

=== Challenger ===
{
  "model_uri": "models:/adult_census_classifier/6",
  "version": "6",
  "metrics": {
    "roc_auc": 0.922008548110852,
    "pr_auc": 0.8188926907038248,
    "accuracy": 0.8674957776754184,
    "precision": 0.7672479150871873,
    "recall": 0.6454081632653061,
    "f1": 0.7010737790093523
  },
  "latency_ms_per_row": 0.04267505860298115,
  "size_bytes": 855054.0
}

--- Decision gate ---
PR-AUC champion   : 0.767865
PR-AUC challenger : 0.818893
PR-AUC gain       : +0.051028 (>= +0.01)
Recall drop       : -0.035714 (

In [31]:
import json
import mlflow

# Make sure we're logging to the registry experiment
mlflow.set_experiment("adult_census_registry")

run_name = f"compare_champion_v{champion_version}_vs_challenger_v{challenger_version}"

with mlflow.start_run(run_name=run_name):
    # Log metrics (both)
    for k, v in champion.items():
        mlflow.log_metric(f"champion_{k}", v)
    for k, v in challenger.items():
        mlflow.log_metric(f"challenger_{k}", v)

    mlflow.log_metric("pr_auc_gain", pr_gain)
    mlflow.log_metric("recall_drop", recall_drop)
    mlflow.log_metric("latency_increase", lat_increase)

    # Log decision policy as params
    mlflow.log_param("min_pr_auc_gain", MIN_PR_AUC_GAIN)
    mlflow.log_param("max_recall_drop", MAX_RECALL_DROP)
    mlflow.log_param("max_latency_increase", MAX_LATENCY_INCREASE)

    # Tags = governance
    mlflow.set_tag("model_name", MODEL_NAME)
    mlflow.set_tag("champion_version", str(champion_version))
    mlflow.set_tag("challenger_version", str(challenger_version))
    mlflow.set_tag("decision", "PASS" if passes else "FAIL")

    # Save a JSON report locally under /reports/registry/
    payload = {
        "run_name": run_name,
        "model_name": MODEL_NAME,
        "champion": champion_info,
        "challenger": challenger_info,
        "gate": {
            "min_pr_auc_gain": MIN_PR_AUC_GAIN,
            "max_recall_drop": MAX_RECALL_DROP,
            "max_latency_increase": MAX_LATENCY_INCREASE,
        },
        "deltas": {
            "pr_auc_gain": pr_gain,
            "recall_drop": recall_drop,
            "latency_increase": lat_increase,
        },
        "passes": bool(passes),
    }

    out_path = REGISTRY_REPORTS_DIR / f"decision_report_v{champion_version}_vs_v{challenger_version}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

    # Log artifact to MLflow (stored under repo/mlflow/artifacts/…)
    mlflow.log_artifact(str(out_path))

print(f"✓ Decision logged to MLflow + saved locally at: {out_path}")

latest_path = REGISTRY_REPORTS_DIR / "decision_report_latest.json"
latest_path.write_text(out_path.read_text(encoding="utf-8"), encoding="utf-8")

✓ Decision logged to MLflow + saved locally at: h:\Documents\2. Perso\github\mlflow\reports\registry\decision_report_v1_vs_v6.json


1188

## Section 11 — Promotion (swap `champion` / `challenger`)

If the challenger passes the gate:
- `@champion` becomes the challenger version
- `@challenger` can be kept (optional), or moved to the next candidate

This creates a clean, reversible, production-friendly workflow.

In [32]:
from mlflow.tracking import MlflowClient
client = MlflowClient()

if passes:
    # Prefer aliases if supported
    if hasattr(client, "set_registered_model_alias"):
        client.set_registered_model_alias(MODEL_NAME, "champion", challenger_version)
        client.set_registered_model_alias(MODEL_NAME, "challenger", champion_version)  # optional: keep old champion as challenger
        
        print("✓ Aliases swapped:")
        print(f"  champion   -> v{challenger_version}")
        print(f"  challenger -> v{champion_version} (old champion)")
        
    else:
        # Fallback: stages
        client.transition_model_version_stage(
            name=MODEL_NAME,
            version=challenger_version,
            stage="Production",
            archive_existing_versions=True,
        )
        print(f"✓ Promoted v{challenger_version} to Production (fallback stages)")
else:
    print("✗ Challenger did NOT pass. No promotion performed.")


✓ Aliases swapped:
  champion   -> v6
  challenger -> v1 (old champion)


## Section 12 — Stable inference entrypoint

In production, your service should always load:
- `models:/name@champion` (preferred, if aliases are supported)
or
- `models:/name/Production` (stage fallback)

Your inference code should NEVER depend on a run_id.

In [26]:
import mlflow.pyfunc

# Prefer champion alias if available, otherwise Production stage
if safe_get_by_alias(MODEL_NAME, "champion"):
    uri = f"models:/{MODEL_NAME}@champion"
else:
    uri = f"models:/{MODEL_NAME}/Production"

prod_model = mlflow.pyfunc.load_model(uri)
print("✓ Loaded production model from:", uri)

# Quick check on a few rows
preds = prod_model.predict(X_test.head(5))
print("Sample preds:", np.asarray(preds))

✓ Loaded production model from: models:/adult_census_classifier@champion
Sample preds: [0 0 1 1 0]
